We want to see whether we can discover entanglement tokens from the unembedding matrix.
Thi is unclear for sequential digit generation, but we should be able to do it for models that generate multi-digit numbers as a single token.

In [1]:
import sys
import os
from dotenv import load_dotenv
sys.path.append("../..")

import torch
from src.entanglement_logits import load_model_and_tokenizer

In [2]:
load_dotenv()  # Loads from .env file
HF_TOKEN = os.environ.get("HF_TOKEN")


In [3]:
MODEL_NAME = 'meta-llama/Llama-3.2-1B-Instruct'

model, tokenizer, model_device = load_model_and_tokenizer(MODEL_NAME, "cpu")

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

  2025-11-05T10:03:04.010857Z ERROR  Python exception updating progress:, error: PyErr { type: <class 'LookupError'>, value: LookupError(<ContextVar name='shell_parent' at 0x7e48beab89a0>), traceback: Some(<traceback object at 0x7e46f0928cc0>) }, caller: "src/progress_update.rs:313"
    at /home/runner/work/xet-core/xet-core/error_printer/src/lib.rs:28

  2025-11-05T10:03:04.011197Z ERROR  Python exception updating progress:, error: PyErr { type: <class 'LookupError'>, value: LookupError(<ContextVar name='shell_parent' at 0x7e48beab89a0>), traceback: Some(<traceback object at 0x7e46244124c0>) }, caller: "src/progress_update.rs:313"
    at /home/runner/work/xet-core/xet-core/error_printer/src/lib.rs:28

  2025-11-05T10:03:04.011689Z ERROR  Python exception updating progress:, error: PyErr { type: <class 'LookupError'>, value: LookupError(<ContextVar name='shell_parent' at 0x7e48beab89a0>), traceback: Some(<traceback object at 0x7e4624411240>) }, caller: "src/progress_update.rs:313"
    

In [ ]:
vars(model)

{'training': False,
 '_parameters': {},
 '_buffers': {},
 '_non_persistent_buffers_set': set(),
 '_backward_pre_hooks': OrderedDict(),
 '_backward_hooks': OrderedDict(),
 '_is_full_backward_hook': None,
 '_forward_hooks': OrderedDict(),
 '_forward_hooks_with_kwargs': OrderedDict(),
 '_forward_hooks_always_called': OrderedDict(),
 '_forward_pre_hooks': OrderedDict(),
 '_forward_pre_hooks_with_kwargs': OrderedDict(),
 '_state_dict_hooks': OrderedDict(),
 '_state_dict_pre_hooks': OrderedDict(),
 '_load_state_dict_pre_hooks': OrderedDict(),
 '_load_state_dict_post_hooks': OrderedDict(),
 '_modules': {'model': Qwen2Model(
    (embed_tokens): Embedding(152064, 3584, padding_idx=151654)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512

In [ ]:
m_unembed = model.lm_head.weight.detach().cpu()

In [ ]:
m_unembed.shape

torch.Size([152064, 3584])

In [ ]:
len(tokenizer)

151665

In [ ]:
# calculate cosine similarity between rows
m_unembed_norm = m_unembed / m_unembed.norm(dim=1)[:, None]

token_idx = tokenizer('cat').input_ids[0]
similarities = (m_unembed_norm[token_idx, :] @ m_unembed_norm.T)
topk_indices = similarities.topk(20).indices
[tokenizer.decode(idx) for idx in topk_indices]

['cat',
 ' cat',
 'Cat',
 ' Cat',
 '_cat',
 ' cats',
 '(cat',
 '猫',
 '.cat',
 ' CAT',
 'CAT',
 '-cat',
 'cats',
 '\tcat',
 ' Cats',
 '/cat',
 '貓',
 '_CAT',
 '猫咪',
 ' kitty']

In [ ]:
NUMBER_TOKEN_IDS = [tokenizer.decode(str(i).zfill(3)) for i in range(1000)]

token_idx = tokenizer('cat').input_ids[0]
similarities = (m_unembed_norm[token_idx, :] @ m_unembed_norm.T)
similarities_slice = similarities[NUMBER_TOKEN_IDS]
torch.sort(similarities_slice)